In [ ]:
!pip install gradio pymupdf sentence-transformers faiss-cpu groq langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.5 MB/s eta 0:00:00


In [ ]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")


Enter your Groq API key: ··········


In [ ]:
import os
import fitz
import faiss
import numpy as np
import gradio as gr

from groq import Groq
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

LLM_MODEL = "llama-3.1-8b-instant"

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Models loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully.


In [ ]:
def extract_pdf_text(pdf_files):
    pages = []

    for pdf_file in pdf_files:
        file_path = pdf_file

        if hasattr(pdf_file, "name"):
            file_path = pdf_file.name

        doc = fitz.open(file_path)

        for page_number, page in enumerate(doc, start=1):
            text = page.get_text()

            if text.strip():
                pages.append({
                    "file_name": file_path.split("/")[-1],
                    "page": page_number,
                    "text": text
                })

    return pages

In [ ]:
def split_text(pages):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    chunks = []

    for page in pages:
        small_chunks = splitter.split_text(page["text"])

        for chunk in small_chunks:
            chunks.append({
                "text": chunk,
                "page": page["page"],
                "file_name": page["file_name"]
            })

    return chunks

In [ ]:
def create_faiss_index(chunks):
    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(texts)
    embeddings = np.array(embeddings).astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index

In [ ]:
def search_relevant_chunks(question, index, chunks, top_k=3):
    question_embedding = embedding_model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indexes = index.search(question_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indexes[0]):
        results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "file_name": chunks[idx]["file_name"],
            "distance": distance
        })

    return results

In [ ]:

def generate_answer(question, retrieved_chunks, chat_history):
    context = "\n\n".join([chunk["text"] for chunk in retrieved_chunks])

    history_text = ""

    for user_msg, bot_msg in chat_history[-3:]:
        history_text += f"User: {user_msg}\n"
        history_text += f"Assistant: {bot_msg}\n\n"

    prompt = f"""
You are a PDF question-answering assistant.

Use the PDF context to answer the current question.
Use previous conversation only to understand follow-up words like it, this, that, he, she, or they.

If the answer is not available in the PDF context, say:
"I could not find this information in the uploaded PDF."

Previous Conversation:
{history_text}

PDF Context:
{context}

Current Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": "You answer only from the provided PDF context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=500
    )

    return response.choices[0].message.content

In [ ]:
stored_chunks = []
stored_index = None
export_chat_text = ""

def process_pdf(pdf_file):
    global stored_chunks, stored_index

    try:
        if pdf_file is None:
            return "Please upload a PDF first."

        print("PDF file received:", pdf_file)
        print("PDF file type:", type(pdf_file))

        pages = extract_pdf_text(pdf_file)
        print("Pages extracted:", len(pages))

        if len(pages) == 0:
            return "PDF text could not be extracted. Try another text-based PDF."

        stored_chunks = split_text(pages)
        print("Chunks created:", len(stored_chunks))

        stored_index = create_faiss_index(stored_chunks)

        return f"PDF processed successfully. Total chunks created: {len(stored_chunks)}"

    except Exception as e:
        return f"Error while processing PDF: {str(e)}"


def chat_with_pdf(question, chat_history):
    global stored_chunks, stored_index, export_chat_text

    try:
        if stored_index is None:
            return "", chat_history, "Please upload and process a PDF first."

        if question.strip() == "":
            return "", chat_history, "Please enter a question."

        retrieved_chunks = search_relevant_chunks(
            question,
            stored_index,
            stored_chunks
        )

        best_distance = retrieved_chunks[0]["distance"]

        if best_distance > 1.6:
            answer = "I could not find this information in the uploaded PDF."
            source = "No source found"
        else:
            answer = generate_answer(question, retrieved_chunks, chat_history)
            sources = sorted(set([
               f"{chunk['file_name']} - Page {chunk['page']}"
               for chunk in retrieved_chunks
]))

source = ", ".join(sources)

        final_answer = f"{answer}\n\nSource: {source}"

        chat_history.append((question, final_answer))

        export_chat_text += f"Question: {question}\n"
        export_chat_text += f"Answer: {answer}\n"
        export_chat_text += f"Source: {source}\n\n"

        return "", chat_history, source

    except Exception as e:
        return "", chat_history, f"Error while answering: {str(e)}"

In [ ]:
def export_chat():
    global export_chat_text

    file_path = "chat_history.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(export_chat_text)

    return file_path

In [ ]:
custom_css = """
body {
    background-color: #0F172A;
}

.gradio-container {
    background: #0F172A !important;
}

/* Title */
h1 {
    color: white !important;
    text-align: center;
}

h2, h3, p, label {
    color: #E2E8F0 !important;
}

/* Input boxes */
textarea,
input {
    color: black !important;
    background-color: white !important;
}

/* Chat messages */
.message,
.message * {
    color: black !important;
}

/* Chatbot container */
.chatbot {
    background-color: white !important;
}

/* Source box */
textarea {
    color: black !important;
}

/* Hide footer */
footer {
    visibility: hidden;
}
"""


with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="blue",
        secondary_hue="slate"
    ),
    css=custom_css
) as app:

    gr.Markdown("""
    # 🤖 AI PDF Assistant
    ### RAG-Based PDF Question Answering System

    Upload your PDF and ask questions using **Groq Llama 3 + FAISS + Conversation Memory**.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(
                label="📤 Upload One or More PDFs",
                file_types=[".pdf"],
                file_count="multiple"
)


            process_button = gr.Button(
                "⚙️ Process PDF",
                variant="primary"
            )

            process_output = gr.Textbox(
                label="Processing Status",
                placeholder="PDF processing status will appear here..."
            )

            source_output = gr.Textbox(
                label="📌 Source Page",
                placeholder="Source page will appear here..."
            )

            export_button = gr.Button(
                "⬇️ Export Chat",
                variant="secondary"
            )

            export_file = gr.File(
                label="Download Chat History",
                interactive=False
            )

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="💬 AI Assistant",
                height=500,
                bubble_full_width=False
            )

            question_input = gr.Textbox(
                label="Ask a question",
                placeholder="Example: Summarize this PDF...",
                lines=2
            )

            answer_button = gr.Button(
                "🚀 Ask AI",
                variant="primary"
            )

    gr.Markdown("""
    ---
    ### ✅ Project Features
    - PDF Question Answering
    - Page Citation
    - Follow-up Question Memory
    - I-don't-know Handling
    - Export Chat History
    """)

    process_button.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=process_output
    )

    answer_button.click(
        fn=chat_with_pdf,
        inputs=[question_input, chatbot],
        outputs=[question_input, chatbot, source_output]
    )

    export_button.click(
        fn=export_chat,
        inputs=[],
        outputs=export_file
    )

app.launch(share=True)